In [1]:
import requests
import feedparser
import time
from pathlib import Path
import pyarrow.parquet as pq

In [2]:
API_URL = "https://export.arxiv.org/api/query"

HEADERS = {
    "User-Agent": "arxiv-category-counter/1.0"
}

In [5]:
CATEGORIES = [
    "cs.AI",
    "cs.LG",
    "cs.CV",
    "cs.CL",
    "cs.IR",
    "cs.NE",
    "cs.RO",
    "cs.DS",
    "cs.DB",
    "cs.CR",
    "math.NA",
    "math.OC",
    "math.PR",
    "math.ST",
    "math.IT",
    "math.CO",
]

PROJECT_ROOT = Path("/home/abdelkarim/project/arxiv-bot")
OUTPUT_DIR = PROJECT_ROOT / "data" / "metadata"

In [6]:
def get_category_count(category):

    params = {
        "search_query" : f"cat:{category}",
        "start" : 0,
        "max_results" : 1
    }

    response = requests.get(
        API_URL,
        params = params,
        headers = HEADERS,
        timeout = 60
    )

    response.raise_for_status()

    feed = feedparser.parse(response.content)

    total = feed.feed.get("opensearch_totalresults")

    if total is None:
        raise ValueError(
            f"Impossible de récupérer le nombre pour {category}"
        )

    return int(total)

In [7]:
results = {}

print("=" * 60)
print("COMPTAGE DES ARTICLES ARXIV")
print("=" * 60)

for category in CATEGORIES:

    print(f"\nRecherche : {category} ...")

    try:
        count = get_category_count(category)

        results[category] = count

        print(f"→ {count:,} articles")

    except Exception as e:
        print(f"Erreur : {e}")

    # Respect de l'API arXiv
    time.sleep(3)


COMPTAGE DES ARTICLES ARXIV

Recherche : cs.AI ...
→ 195,935 articles

Recherche : cs.LG ...
→ 282,358 articles

Recherche : cs.CV ...
→ 202,995 articles

Recherche : cs.CL ...
→ 116,877 articles

Recherche : cs.IR ...
→ 26,718 articles

Recherche : cs.NE ...
→ 18,159 articles

Recherche : cs.RO ...
→ 57,052 articles

Recherche : cs.DS ...
→ 29,703 articles

Recherche : cs.DB ...
→ 12,240 articles

Recherche : cs.CR ...
→ 51,227 articles

Recherche : math.NA ...
→ 51,095 articles

Recherche : math.OC ...
→ 63,643 articles

Recherche : math.PR ...
→ 67,461 articles

Recherche : math.ST ...
→ 28,733 articles

Recherche : math.IT ...
→ 0 articles

Recherche : math.CO ...
→ 81,239 articles


In [8]:
print("\n")
print("=" * 60)
print("NOMBRE D'ARTICLES PAR CATÉGORIE")
print("=" * 60)

total = 0

for category, count in results.items():

    print(f"{category:<10} : {count:>12,}")

    total += count

print("=" * 60)
print(f"{'TOTAL':<10} : {total:>12,}")
print("=" * 60)



NOMBRE D'ARTICLES PAR CATÉGORIE
cs.AI      :      195,935
cs.LG      :      282,358
cs.CV      :      202,995
cs.CL      :      116,877
cs.IR      :       26,718
cs.NE      :       18,159
cs.RO      :       57,052
cs.DS      :       29,703
cs.DB      :       12,240
cs.CR      :       51,227
math.NA    :       51,095
math.OC    :       63,643
math.PR    :       67,461
math.ST    :       28,733
math.IT    :            0
math.CO    :       81,239
TOTAL      :    1,285,435


In [7]:
ids = set()
for f in OUTPUT_DIR.glob("batch_*.parquet"):
    import pyarrow.parquet as pq
    table = pq.read_table(f, columns=["arxiv_id"])
    ids.update(table["arxiv_id"].to_pylist())

print(f"NOMBRE D'ARTICLES STOCKER : {len(ids)}")

NOMBRE D'ARTICLES STOCKER : 26365
